# Nemotron-Reward 32B: быстрый инференс (RU пары)

Этот ноутбук считает **reward score** для пары *(prompt, answer)* и сравнивает два ответа по разнице score.

Модели:
- `nvidia/Qwen-3-Nemotron-32B-Reward`
- `nvidia/Qwen-2.5-Nemotron-32B-Reward`

Ресурсы: 1× **A100 80GB**. По умолчанию грузим **bf16** без квантизации (batch=1). Если вдруг не влезет — есть fallback на 8-bit.

In [ ]:
import os
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# общий HF-кэш (как на DGX Lab)
os.environ.setdefault("HF_HOME", "/shared/hf_cache")
os.environ.setdefault("HF_HUB_CACHE", os.path.join(os.environ["HF_HOME"], "hub"))

assert torch.cuda.is_available(), "Нужна GPU (CUDA)"
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Выберите модель
MODEL_ID = "nvidia/Qwen-3-Nemotron-32B-Reward"
# MODEL_ID = "nvidia/Qwen-2.5-Nemotron-32B-Reward"

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("dtype:", DTYPE)

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Пытаемся загрузить без квантизации (A100 80GB обычно хватает для 32B bf16 при batch=1)
try:
    rm = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=DTYPE,
        device_map="auto",
        num_labels=1,
    )
    _quant = "none"
except RuntimeError as e:
    print("BF16/FP16 load failed, trying 8-bit (bitsandbytes). Error:", e)
    from transformers import BitsAndBytesConfig

    bnb = BitsAndBytesConfig(load_in_8bit=True)
    rm = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb,
        device_map="auto",
        num_labels=1,
    )
    _quant = "8bit"

rm.eval()
print("loaded:", MODEL_ID, "| quant:", _quant)

In [ ]:
def _to_chat_text(prompt: str, answer: str) -> str:
    # Универсально: если у токенизатора есть chat template — используем его.
    if getattr(tok, "chat_template", None):
        msgs = [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}]
        return tok.apply_chat_template(msgs, tokenize=False)

    # Иначе — простой конкат (хуже, но работает как fallback)
    return f"User: {prompt}\nAssistant: {answer}"


@torch.no_grad()
def score(prompt: str, answer: str, max_length: int = 2048) -> float:
    text = _to_chat_text(prompt, answer)
    enc = tok(text, return_tensors="pt", truncation=True, max_length=max_length)
    # device_map=auto => параметры могут быть на разных устройствах; вход оставим на cuda:0
    enc = {k: v.to("cuda") for k, v in enc.items()}
    out = rm(**enc)
    return float(out.logits[0, 0].item())


def prefer(prompt: str, a: str, b: str) -> dict:
    sa = score(prompt, a)
    sb = score(prompt, b)
    return {
        "score_a": sa,
        "score_b": sb,
        "margin_a_minus_b": sa - sb,
        "winner": "A" if sa > sb else ("B" if sb > sa else "tie"),
    }

# мини-тест
prefer("Объясни, что такое градиентный спуск.", "Коротко: ...", "Подробно: ...")

In [ ]:
# Прогон ~200 запросов из вашего датасета (формат как в skywork ноутбуке)
DATA_DIR = Path("..") / "dataset" / "splitted"
TEST_PATH = DATA_DIR / "test_ds.json"

raw = json.loads(TEST_PATH.read_text(encoding="utf-8"))

pairs = []
for ex in raw:
    w = ex.get("agg_label")
    if w == "model":
        chosen, rejected = ex["model_answer"], ex["reference_answer"]
    elif w == "reference":
        chosen, rejected = ex["reference_answer"], ex["model_answer"]
    else:
        continue
    pairs.append((ex["instruction"], chosen, rejected))

N = min(200, len(pairs))
print("pairs:", len(pairs), "running:", N)

out = []
correct = 0
for i in range(N):
    prompt, chosen, rejected = pairs[i]
    sc = score(prompt, chosen)
    sr = score(prompt, rejected)
    ok = int(sc > sr)
    correct += ok
    out.append({
        "i": i,
        "margin": sc - sr,
        "ok": ok,
        "score_chosen": sc,
        "score_rejected": sr,
    })

print("pairwise acc:", correct / N if N else None)
print("margin mean:", sum(r["margin"] for r in out) / N if N else None)

# сохранить результаты
OUT_JSON = Path("nemotron_scores_200.json")
OUT_JSON.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", OUT_JSON.resolve())